# Study D, Milestone 4
Dataset expansion, audited Fisher information, full-tree dependence, calibration, the capacity law verified against real tree geometry, and the Quinn replication. See `reports/milestone4_decision.md`.

In [1]:
import json, os, sys, numpy as np, pandas as pd
sys.path.insert(0, '..')
pd.set_option('display.width', 220)
T = '../results/tables/'
have = lambda f: os.path.exists(T + f)

## 1. Fisher-information audit: the Milestone 3 optimum restated

In [2]:
o = pd.read_csv(T+'e12_optimum_audit.csv')
display(o.round(4))
print('kL* (pi known):  mean %.3f  range %.3f-%.3f' % (o.kL_opt_pi_known.mean(),
      o.kL_opt_pi_known.min(), o.kL_opt_pi_known.max()))
print('kL* (pi joint):  mean %.3f  range %.3f-%.3f' % (o.kL_opt_pi_joint.mean(),
      o.kL_opt_pi_joint.min(), o.kL_opt_pi_joint.max()))
print('worst-case information cost of estimating pi: %.1f%%' %
      (100*(1-o.ratio_joint_over_known.min())))

,k,pi,kL_opt_pi_known,kL_opt_pi_joint,I_max_known,I_max_joint,ratio_joint_over_known
0,0.05,0.1,0.4838,0.4989,0.0653,0.0611,0.9358
1,0.05,0.3,0.4152,0.4152,0.1393,0.1370,0.9832
2,0.05,0.5,0.4027,0.4027,0.1619,0.1619,1.0000
3,0.05,0.7,0.4152,0.4152,0.1393,0.1370,0.9832
4,0.10,0.1,0.4788,0.4937,0.0653,0.0611,0.9357
5,0.10,0.3,0.4109,0.4109,0.1393,0.1369,0.9832
6,0.10,0.5,0.3985,0.3985,0.1619,0.1619,1.0000
7,0.10,0.7,0.4109,0.4109,0.1393,0.1369,0.9832
8,0.50,0.1,0.4878,0.5030,0.0653,0.0611,0.9358
9,0.50,0.3,0.4186,0.4186,0.1393,0.1370,0.9833


kL* (pi known):  mean 0.426  range 0.394-0.488
kL* (pi joint):  mean 0.431  range 0.394-0.503
worst-case information cost of estimating pi: 6.4%


In [3]:
w = pd.read_csv(T+'e12_optimum_width.csv')
print('kL band retaining >=50%% of peak: %.2f - %.2f (%.1f-fold)' %
      (w.kL_lo_50pct.median(), w.kL_hi_50pct.median(), w.fold_width.median()))

kL band retaining >=50% of peak: 0.10 - 0.99 (9.6-fold)


## 2. Full-tree information versus the naive independent-pair sum

In [4]:
if have('e13_tree_vs_pair_mc.csv'):
    d = pd.read_csv(T+'e13_tree_vs_pair_mc.csv')
    display(d.groupby('depth').inflation.describe()[['count','min','50%','max']].round(2))
    display(d.groupby('k').inflation.median().round(2).rename('median inflation'))
    v = d.dropna(subset=['I_exact_logk'])
    print('MC vs exact enumeration: max relative error %.4f over %d settings' %
          (((v.I_tree_logk-v.I_exact_logk).abs()/v.I_exact_logk).max(), len(v)))

,count,min,50%,max
depth,,,,
3,12.0,1.00,3.39,8.30
4,12.0,0.98,5.17,20.16
5,12.0,0.95,7.35,45.79
6,12.0,0.96,10.18,98.98
7,12.0,0.98,14.17,221.27


k
0.05    45.73
0.10    30.74
0.20    13.04
0.50     1.99
1.00     1.06
2.00     0.98
Name: median inflation, dtype: float64

MC vs exact enumeration: max relative error 0.0599 over 12 settings


## 3. Calibration (300 replicates per scenario, Wilson intervals)

In [5]:
if have('e14_calibration_summary.csv'):
    s = pd.read_csv(T+'e14_calibration_summary.csv')
    print('replicates per scenario:', int(s.n.iloc[0]))
    display(s.pivot(index='scen', columns='test', values='rate').round(3))
    lo = s.pivot(index='scen', columns='test', values='lo')
    hi = s.pivot(index='scen', columns='test', values='hi')
    for sc in [x for x in ['H0_pi','H0_k','H0_k_frozen','MIS'] if x in lo.index]:
        print(sc, {t: '%.3f [%.3f, %.3f]' % (s.pivot(index='scen',columns='test',values='rate').loc[sc,t],
              lo.loc[sc,t], hi.loc[sc,t]) for t in ['chi2_pi','chi2_k','cover_V_pi'] if t in lo})

replicates per scenario: 300


test,calib_k,calib_pi,chi2_k,chi2_pi,cover_V_pi,cover_pi_profile
scen,,,,,,
H0_k,0.050,0.517,0.013,0.903,0.920,0.918
H0_k_frozen,0.010,0.967,0.010,1.000,0.253,0.965
H0_pi,0.897,0.050,0.833,0.143,0.967,0.935
H1_k,0.883,0.047,0.830,0.163,0.947,0.936
H1_pi,0.077,0.493,0.033,0.867,0.880,0.913
MIS,0.073,0.003,0.040,0.123,0.990,0.719


H0_pi {'chi2_pi': '0.143 [0.108, 0.188]', 'chi2_k': '0.833 [0.787, 0.871]', 'cover_V_pi': '0.967 [0.940, 0.982]'}
H0_k {'chi2_pi': '0.903 [0.865, 0.932]', 'chi2_k': '0.013 [0.005, 0.034]', 'cover_V_pi': '0.920 [0.884, 0.946]'}
H0_k_frozen {'chi2_pi': '1.000 [0.987, 1.000]', 'chi2_k': '0.010 [0.003, 0.029]', 'cover_V_pi': '0.253 [0.207, 0.305]'}
MIS {'chi2_pi': '0.123 [0.091, 0.165]', 'chi2_k': '0.040 [0.023, 0.069]', 'cover_V_pi': '0.990 [0.971, 0.997]'}


In [6]:
if have('e14_thresholds.csv'):
    display(pd.read_csv(T+'e14_thresholds.csv').round(2))

,scen,thr_boot,thr_chi2
0,H0_pi,32.59,19.68
1,H0_k,17.35,19.68
2,H0_k_frozen,9.81,19.68


## 4. The capacity law, verified against real tree geometry

In [7]:
if have('e16_capacity_vs_prediction.csv'):
    e = pd.read_csv(T+'e16_capacity_vs_prediction.csv')
    display(e.round(3))
    print('median |empirical - predicted| decades: %.3f' % e.abs_error.median())

,system,dataset,clones,depth_max,L_min,L_max,k_lo,k_hi,decades,decades_predicted,abs_error
0,Cassiopeia,m100k,19,9.0,3.310,7.831,0.013,0.302,1.275,1.954,0.679
1,Cassiopeia,m10k,11,8.0,4.260,6.868,0.015,0.235,1.185,1.903,0.718
2,Cassiopeia,m30k,4,13.0,9.189,11.813,0.008,0.110,1.106,2.114,1.007
3,Cassiopeia,m5k,82,20.0,10.814,19.000,0.005,0.092,1.243,2.301,1.058
4,macsGESTALT,mouse1,12,4.0,1.212,2.861,0.035,0.828,1.374,1.602,0.228
5,macsGESTALT,mouse2,24,3.5,0.500,1.924,0.052,2.000,1.354,1.544,0.190


median |empirical - predicted| decades: 0.699


In [8]:
if have('e15b_capacity_designs.csv'):
    d = pd.read_csv(T+'e15b_capacity_designs.csv')
    display(d.groupby(['S','D']).decades_covered.first().unstack().round(2))
    print('Pareto-optimal designs:', int(d.pareto.sum()), 'of', len(d))

D,6,8,10,12
S,,,,
2,1.30,1.3,1.3,1.30
4,1.60,1.6,1.6,1.60
8,1.78,1.9,1.9,1.90
16,1.78,1.9,2.0,2.08
32,1.78,1.9,2.0,2.08


Pareto-optimal designs: 5 of 100


## 5. Quinn independent system: four animals

In [9]:
for f, lab in [('m4_quinn_summary.csv','cohort'),
               ('m4_quinn_variance_components.csv','all sites, cell-cycle state'),
               ('m4_quinn_variance_components_pc1.csv','all sites, PC1 sensitivity'),
               ('m4_quinn_primary_only.csv','primary site only, depth-adjusted')]:
    if have(f):
        print('==', lab); display(pd.read_csv(T+f).round(4))

== cohort


,mouse,state,n_cells,n_sites,n_lineage_groups,n_trees_usable,cells_in_trees,primary_cells
0,5k,state2_cc,37814,6,100,74,11846,11106
1,30k,state2_cc,17189,6,4,3,1031,1588
2,10k,state2_cc,2541,9,13,11,2365,930
3,100k,state2_cc,5806,6,31,15,2045,4430


== all sites, cell-cycle state


,mouse,state,scope,lrt_pi_stat,lrt_pi_p,lrt_k_stat,lrt_k_p,n_clones,V_pi_raw,V_pi_noise,V_pi,mean_pi,n_clones_k_identified,V_logk_raw,V_logk_noise,V_logk,median_k
0,5k,state2_cc,all_sites,164.0193,0.0000,59.4778,0.0007,30,0.0084,0.0012,0.0072,0.3846,28,1.1880,0.5344,0.6536,0.6455
1,30k,state2_cc,all_sites,7.0426,0.0296,3.0072,0.2223,3,0.0051,0.0008,0.0043,0.4410,3,0.1021,0.5792,0.0000,0.7566
2,10k,state2_cc,all_sites,56.0251,0.0000,8.6999,0.5608,11,0.0147,0.0044,0.0103,0.5418,11,1.5084,0.9942,0.5143,0.9383
3,100k,state2_cc,all_sites,162.2647,0.0000,31.7410,0.0044,15,0.0504,0.0075,0.0429,0.5342,13,2.2218,1.2869,0.9349,0.9026


== all sites, PC1 sensitivity


,mouse,state,scope,lrt_pi_stat,lrt_pi_p,lrt_k_stat,lrt_k_p,n_clones,V_pi_raw,V_pi_noise,V_pi,mean_pi,n_clones_k_identified,V_logk_raw,V_logk_noise,V_logk,median_k
0,5k,state2,all_sites,153.9092,0.0000,52.6878,0.0046,30,0.0064,0.0011,0.0053,0.4965,25,0.5469,0.4774,0.0694,0.6262
1,30k,state2,all_sites,17.7828,0.0001,5.1670,0.0755,3,0.0191,0.0009,0.0182,0.6421,3,0.1199,0.3451,0.0000,0.5793
2,10k,state2,all_sites,67.8804,0.0000,50.3220,0.0000,11,0.0166,0.0119,0.0047,0.5719,10,0.5827,0.8751,0.0000,0.5422
3,100k,state2,all_sites,91.4084,0.0000,27.2863,0.0177,15,0.0199,0.0085,0.0114,0.6428,12,0.4976,0.8315,0.0000,0.4904


== primary site only, depth-adjusted


,mouse,scope,lrt_pi_stat,lrt_pi_p,lrt_k_stat,lrt_k_p,n_clones,V_pi_raw,V_pi_noise,V_pi,mean_pi,n_clones_k_identified,V_logk_raw,V_logk_noise,V_logk,median_k
0,5k,primary_only_depth_adjusted,45.8640,0.0242,25.5670,0.6485,30,0.0071,0.0050,0.0021,0.4890,21,1.2447,1.0837,0.1610,0.7198
1,10k,primary_only_depth_adjusted,59.2985,0.0000,1.2207,0.9759,7,0.0357,0.0069,0.0288,0.6103,7,0.5576,1.3363,0.0000,1.0928
2,100k,primary_only_depth_adjusted,88.9863,0.0000,24.0384,0.0042,10,0.0356,0.0036,0.0321,0.5864,8,2.3754,0.7110,1.6644,0.8981


### Negative control: within-clone shifts against quality covariates

In [10]:
if have('m4_quinn_technical_control.csv'):
    t = pd.read_csv(T+'m4_quinn_technical_control.csv')
    display(t.pivot_table(index=['mouse','site'], columns='feature',
                          values='within_clone_shift_sd').round(3))

feature     cc_score    lib  mt_frac  n_genes    pc1
mouse site                                          
100k  MS      -0.027  0.139    0.044    0.124  0.023
10k   M1      -0.109  0.074   -0.093    0.045  0.409
      M2      -0.021  0.357    0.247    0.321  0.477
30k   BL       0.061 -0.262    0.850   -0.241 -0.153
      LN      -0.017  0.013    0.077   -0.027 -0.079
      M1      -0.210 -0.109    0.323   -0.092  0.190
      M2      -0.342 -0.210   -0.045   -0.221 -0.074
5k    Liv      0.337  0.662    0.086    0.588  0.056
      M1      -0.641 -0.947   -0.074   -1.390  0.134
      M2      -0.483 -0.851   -0.002   -1.100  0.094